# Phase 1 — Colab 基线与 CPPO 放大（g8）

对应计划：[schedule.md](../schedule.md) **Phase 1**。

**目标**：在同一 Drive 路径上跑通 g8 对比，争取相对 A2 获得 **≥10%** 训练加速，且 reward 不明显下降。

| 策略 | 含义 | 输出目录 |
|------|------|----------|
| **A2** | GRPO baseline（`cppo_pruning_rate=0`） | `outputs/grpo_A_g8` |
| **C2** | CPPO `p=0.5` | `outputs/grpo_CPPO_g8_p50` |
| **C3** | CPPO `p=0.75`（可选） | `outputs/grpo_CPPO_g8_p75` |

| 项 | 默认 |
|----|------|
| Drive | `/content/drive/MyDrive/MixUpLLaVA-video-r1` |
| `NUM_GENERATIONS` | **8** |
| `MAX_STEPS` | 默认 **50（快验）**；正式可改 100 |
| `SAVE_STEPS` | **10**（断点续训，省 Drive） |
| 进度 | 流式日志 + 简易进度条 / ETA |
| 输出 | `outputs/grpo_A_g8`、`grpo_CPPO_g8_p50`、`grpo_CPPO_g8_p75` |

**成本**：A100 贵，优先只跑 A2 + C2；确认加速后再考虑 C3。

**断点续训**：A2/C2/C3 共用 §6；中断后重连 → 按顺序重跑 §0–§6 → 再跑对应训练单元即可自动 resume（若存在 `checkpoint-*`）。


---
## ⚠️ 每次重连 Colab 后的顺序

| # | 单元 | 说明 |
|---|------|------|
| 0 | 挂载 Drive | 授权 |
| 1 | 环境检查 | 确认 A100 / Drive 路径 |
| 2 | 路径配置 | `PROJECT_DIR`、`SAVE_STEPS`、`ENABLE_RESUME` |
| 2.5 | Phase 0 | 四目录 + `git pull` MixUp |
| 3–4 | 数据 / 依赖 | 小数据集 + pip（已装可快过） |
| 5 | patches | 覆盖 trainer；A2 前确保 prune=0 |
| 6 | 工具函数 | **必须重跑**（进度条 + resume） |
| 7–9 | A2 → C2 → C3 | 有 checkpoint 会自动续训 |

**息屏**：训练在 Colab GPU，但浏览器断连仍可能杀会话。建议 `caffeinate -dims`（Mac）并保持标签页打开。


---
## 1. 环境与路径检查


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# 环境与路径检查
import sys
import os

def check_path(p, name):
    exists = os.path.exists(p)
    print(f"  [{name}] {p}  ->  {'存在' if exists else '不存在'}")
    if exists and os.path.isdir(p):
        try:
            print(f"        子项(前5个): {os.listdir(p)[:5]}")
        except Exception as e:
            print(f"        listdir 失败: {e}")
    return exists

print("=== Python ===")
print(sys.version)
print("\n=== 是否 Colab ===")
try:
    import google.colab
    print("是 Colab")
    IN_COLAB = True
except ImportError:
    print("否（本地 Jupyter）")
    IN_COLAB = False

print("\n=== GPU ===")
try:
    import torch
    print(f"PyTorch: {torch.__version__}, CUDA 可用: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"  设备: {name}, 显存(GB): {mem:.2f}")
        if "A100" in name and mem > 70:
            print("  [建议] A100 高配，可使用 NUM_GENERATIONS=8")
        else:
            print("  [注意] 非 A100 大显存，建议 NUM_GENERATIONS=4")
except Exception as e:
    print(f"  {e}")

print("\n=== Drive 顶层（Phase 1 根目录） ===")
root = "/content/drive/MyDrive/MixUpLLaVA-video-r1"
for sub in ["", "repo", "data/dataset", "checkpoints/coldstart", "outputs"]:
    check_path(os.path.join(root, sub) if sub else root, sub or "PROJECT_DIR")


=== Python ===
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

=== 是否 Colab ===
是 Colab

=== GPU ===
PyTorch: 2.11.0+cu128, CUDA 可用: True
  设备: NVIDIA A100-SXM4-80GB, 显存(GB): 85.09
  [建议] A100 高配，可使用 NUM_GENERATIONS=8

=== Drive 顶层（Phase 1 根目录） ===
  [PROJECT_DIR] /content/drive/MyDrive/MixUpLLaVA-video-r1  ->  存在
        子项(前5个): ['repo', 'data', 'outputs', 'checkpoints']
  [repo] /content/drive/MyDrive/MixUpLLaVA-video-r1/repo  ->  存在
        子项(前5个): ['TinyLLaVA-Video-R1', 'TinyLLaVA-Video-R1-CPPO']
  [data/dataset] /content/drive/MyDrive/MixUpLLaVA-video-r1/data/dataset  ->  存在
        子项(前5个): ['.cache', '.gitattributes', 'README.md', 'nextqa-coldstart-16.json', 'nextqa_0-30s.jsonl']
  [checkpoints/coldstart] /content/drive/MyDrive/MixUpLLaVA-video-r1/checkpoints/coldstart  ->  存在
        子项(前5个): ['.cache', 'merges.txt', 'generation_config.json', 'README.md', '.gitattributes']
  [outputs] /content/drive/MyDrive/MixUpLLaVA-video-r1/outputs  ->  存在
        子项(前5个): ['grpo_A_bas

---
## 2. 路径配置（Phase 1 超参）

默认 **快验**：`MAX_STEPS=50`。正式对比把 `MAX_STEPS` 改为 `100`。

**耗时参考**：v1（g=4、50 step）合计约 29 min；A2（g=8）经验约 **45–90 min**（实际约 ~2.4 min/step）。

**断点续训（A2/C2/C3）**：
- `SAVE_STEPS=10`：约每 10 step 写 `checkpoint-*`（保留最近 **1** 个，省云盘）
- `ENABLE_RESUME=True`：自动从最新 checkpoint 续训（A2 当前应从 `checkpoint-40` 续到 50）
- **跑前建议**：Drive 清空回收站；`grpo_A_g8` 可只留 `checkpoint-40`，删掉 30/35 再开跑


In [ ]:
# ========== Phase 1 路径与超参 ==========
import os

PROJECT_DIR = "/content/drive/MyDrive/MixUpLLaVA-video-r1"
REPO_NAME = "TinyLLaVA-Video-R1"
REPO = os.path.join(PROJECT_DIR, "repo", REPO_NAME)
MIXUP_REPO = os.path.join(PROJECT_DIR, "repo", "MixUpLLaVA-Video-R1")

DATA_ROOT = os.path.join(PROJECT_DIR, "data", "dataset").rstrip("/") + "/"
DATA_ROOT_STRIP = os.path.join(PROJECT_DIR, "data", "dataset")
_ten_p = os.path.join(DATA_ROOT_STRIP, "nextqa_0-30s_10p_seed42.jsonl")
DATA_JSONL = _ten_p if os.path.exists(_ten_p) else os.path.join(DATA_ROOT_STRIP, "nextqa_0-30s.jsonl")
CKPT = os.path.join(PROJECT_DIR, "checkpoints", "coldstart")
OUT_BASE = os.path.join(PROJECT_DIR, "outputs")

# Phase 1 固定（schedule §1.1）
SMALL_N = 50
SMALL_JSONL = os.path.join(DATA_ROOT_STRIP, f"nextqa_small{SMALL_N}.jsonl")
NUM_GENERATIONS = 8      # A100: 8；T4: 4
MAX_STEPS = 50           # 快验 50；正式改为 100
NUM_FRAMES = 2
NUM_QUERIES = 32
MODEL_MAX_LENGTH = 256
NUM_FRAME_TRAINER = 8    # trainer 内采样帧数（与前期一致）

# 断点续训 + 进度（A2/C2/C3 共用，须配合 §6）
SAVE_STEPS = 10          # 每 N step 存盘；10 更省 Drive（原 5 易占满）
ENABLE_RESUME = True     # 自动从最新 checkpoint-* 续训
LOGGING_STEPS = 1        # 密日志，便于解析进度
SAVE_TOTAL_LIMIT = 1     # 只保留最新 1 个 checkpoint，避免云盘爆满

OUT_A2 = os.path.join(OUT_BASE, "grpo_A_g8")
OUT_C2 = os.path.join(OUT_BASE, "grpo_CPPO_g8_p50")
OUT_C3 = os.path.join(OUT_BASE, "grpo_CPPO_g8_p75")

print("PROJECT_DIR:", PROJECT_DIR, "->", os.path.isdir(PROJECT_DIR))
print("REPO:", REPO, "->", os.path.isdir(REPO))
print("MIXUP_REPO:", MIXUP_REPO, "->", os.path.isdir(MIXUP_REPO))
print("DATA_JSONL:", DATA_JSONL, "->", os.path.exists(DATA_JSONL))
print("CKPT:", CKPT, "->", os.path.isdir(CKPT))
print("OUT_BASE:", OUT_BASE)
print(f"NUM_GENERATIONS={NUM_GENERATIONS} | MAX_STEPS={MAX_STEPS} | NUM_FRAMES={NUM_FRAMES}")
print(f"SAVE_STEPS={SAVE_STEPS} | ENABLE_RESUME={ENABLE_RESUME} | LOGGING_STEPS={LOGGING_STEPS}")
print("OUT_A2 / C2 / C3:", OUT_A2, "|", OUT_C2, "|", OUT_C3)


---
## 2.5 Phase 0 验收 + 同步 MixUp 仓库（首次必跑）

若 `MIXUP_REPO` 不存在，下面会 `git clone`；已存在则 `git pull`。


In [4]:
# Phase 0.2 / 0.3：四目录检查 + MixUp 仓库同步
import os
import subprocess

required = {
    "repo": os.path.join(PROJECT_DIR, "repo"),
    "data": os.path.join(PROJECT_DIR, "data"),
    "checkpoints": os.path.join(PROJECT_DIR, "checkpoints"),
    "outputs": os.path.join(PROJECT_DIR, "outputs"),
    "TinyLLaVA REPO": REPO,
    "coldstart": CKPT,
}
ok = True
for k, p in required.items():
    exists = os.path.isdir(p)
    print(f"[{'OK' if exists else 'MISSING'}] {k}: {p}")
    ok = ok and exists
if not ok:
    raise RuntimeError("Drive 结构不完整。请确认已「添加到云端硬盘」共享包，且 PROJECT_DIR 正确。")

repo_parent = os.path.join(PROJECT_DIR, "repo")
os.makedirs(repo_parent, exist_ok=True)
if not os.path.isdir(MIXUP_REPO):
    print("克隆 MixUpLLaVA-Video-R1 ...")
    subprocess.run(
        ["git", "clone", "https://github.com/SanXue-YG/MixUpLLaVA-Video-R1.git", MIXUP_REPO],
        check=True,
    )
else:
    print("更新 MixUpLLaVA-Video-R1 (git pull) ...")
    subprocess.run(["git", "pull"], cwd=MIXUP_REPO, check=False)

patches = os.path.join(MIXUP_REPO, "mixup", "patches")
assert os.path.isdir(patches), f"缺少 mixup/patches: {patches}（请确认 git pull 成功）"
print("Phase 0 验收通过。patches:", os.listdir(patches))


[OK] repo: /content/drive/MyDrive/MixUpLLaVA-video-r1/repo
[OK] data: /content/drive/MyDrive/MixUpLLaVA-video-r1/data
[OK] checkpoints: /content/drive/MyDrive/MixUpLLaVA-video-r1/checkpoints
[OK] outputs: /content/drive/MyDrive/MixUpLLaVA-video-r1/outputs
[OK] TinyLLaVA REPO: /content/drive/MyDrive/MixUpLLaVA-video-r1/repo/TinyLLaVA-Video-R1
[OK] coldstart: /content/drive/MyDrive/MixUpLLaVA-video-r1/checkpoints/coldstart
克隆 MixUpLLaVA-Video-R1 ...
Phase 0 验收通过。patches: ['README.md', 'tinyllava_trainer_reason_baseline.py', 'tinyllava_trainer_reason_cppo.py', 'tinyllava_trainer_reason_strategy_b.py']


---
## 3. 准备小规模数据集


In [5]:
import itertools

if not os.path.exists(DATA_JSONL):
    raise FileNotFoundError(f"请先准备数据: {DATA_JSONL}")

with open(DATA_JSONL, "r", encoding="utf-8") as rf, open(SMALL_JSONL, "w", encoding="utf-8") as wf:
    for rec in itertools.islice(rf, SMALL_N):
        wf.write(rec)
print(f"已写入 {SMALL_N} 条到 {SMALL_JSONL}")


已写入 50 条到 /content/drive/MyDrive/MixUpLLaVA-video-r1/data/dataset/nextqa_small50.jsonl


---
## 4. 确认上游 REPO 并安装依赖


In [6]:
# 确认 TinyLLaVA REPO
repo_parent = os.path.join(PROJECT_DIR, "repo")
if not os.path.isdir(REPO):
    alt = os.path.join(repo_parent, "TinyLLaVA-Video-R1-main")
    if os.path.isdir(alt):
        REPO = alt
assert os.path.isdir(REPO), f"REPO 不存在: {REPO}"
print("当前 REPO:", REPO)


当前 REPO: /content/drive/MyDrive/MixUpLLaVA-video-r1/repo/TinyLLaVA-Video-R1


In [7]:
# 安装依赖
import subprocess
import sys

cmds = [
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"],
    [sys.executable, "-m", "pip", "install", "-q", "trl", "datasets", "pytorchvideo", "decord",
     "transformers", "accelerate", "deepspeed"],
]
for cmd in cmds:
    subprocess.run(cmd, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], cwd=REPO, check=True)

try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "math_verify"], check=True)
except Exception as e:
    print("math_verify 安装失败:", e)

try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flash-attn==2.7.3", "--no-build-isolation"], check=True)
    print("flash-attn 安装成功")
except Exception as e:
    print("flash-attn 未安装，训练将用 eager:", e)


flash-attn 安装成功


---
## 5. 应用 MixUp patches → 上游 trainer（A2 基线：g8 + CPPO off）

从 `MIXUP_REPO/mixup/patches/tinyllava_trainer_reason_cppo.py` 覆盖到 REPO（含 CPPO 字段），再把 `cppo_pruning_rate` 置 0，并设置 `num_generations`。


In [8]:
# 覆盖 trainer 并配置为 A2（GRPO，无剪枝）
import re
import shutil

src_patch = os.path.join(MIXUP_REPO, "mixup", "patches", "tinyllava_trainer_reason_cppo.py")
trainer_path = os.path.join(REPO, "tinyllava", "train", "tinyllava_trainer_reason.py")
assert os.path.isfile(src_patch), f"缺少补丁: {src_patch}"
shutil.copy(src_patch, trainer_path)

with open(trainer_path, "r", encoding="utf-8") as f:
    content = f.read()

content = re.sub(r"self\.num_generations = .*", f"self.num_generations = {NUM_GENERATIONS}  # Phase1 A2/C2", content, count=1)
content = re.sub(r"self\.num_frame = .*", f"self.num_frame = {NUM_FRAME_TRAINER}", content, count=1)
content = re.sub(
    r"self\.cppo_pruning_rate = .*",
    "self.cppo_pruning_rate = 0.0  # A2 GRPO baseline",
    content,
    count=1,
)

with open(trainer_path, "w", encoding="utf-8") as f:
    f.write(content)

print("已覆盖并配置 A2:", trainer_path)
print(f"  num_generations={NUM_GENERATIONS}, num_frame={NUM_FRAME_TRAINER}, cppo_pruning_rate=0.0")


已覆盖并配置 A2: /content/drive/MyDrive/MixUpLLaVA-video-r1/repo/TinyLLaVA-Video-R1/tinyllava/train/tinyllava_trainer_reason.py
  num_generations=8, num_frame=8, cppo_pruning_rate=0.0


---
## 6. 训练工具函数（A2/C2/C3 共用）

功能：ZeRO-3 配置、流式日志、进度条/ETA、按 `SAVE_STEPS` 存盘、自动 resume。

重连后 **必须重新运行本单元**，再跑 §7/§8/§9。


In [ ]:
import os
import json
import re
import shutil
import subprocess
import time
from pathlib import Path

def ensure_zero3_config(repo):
    ds_scripts = os.path.join(repo, "scripts")
    zero3_offload_path = os.path.join(ds_scripts, "zero3_offload.json")
    os.makedirs(ds_scripts, exist_ok=True)
    with open(zero3_offload_path, "w") as f:
        json.dump({
            "fp16": {"enabled": "auto", "loss_scale": 0, "loss_scale_window": 1000,
                     "initial_scale_power": 16, "hysteresis": 2, "min_loss_scale": 1},
            "bf16": {"enabled": "auto"},
            "zero_optimization": {
                "stage": 3,
                "offload_optimizer": {"device": "none", "pin_memory": True},
                "offload_param": {"device": "cpu", "pin_memory": True},
                "overlap_comm": True, "contiguous_gradients": True,
                "sub_group_size": 1000000000, "reduce_bucket_size": "auto",
                "stage3_prefetch_bucket_size": "auto",
                "stage3_param_persistence_threshold": "auto",
                "stage3_max_live_parameters": 1000000000,
                "stage3_max_reuse_distance": 1000000000,
                "stage3_gather_16bit_weights_on_model_save": True,
            },
            "gradient_accumulation_steps": "auto", "gradient_clipping": "auto",
            "steps_per_print": 100, "train_batch_size": "auto",
            "train_micro_batch_size_per_gpu": "auto",
            "wall_clock_breakdown": False,
        }, f, indent=2)
    return zero3_offload_path

def list_checkpoints(output_dir):
    if not os.path.isdir(output_dir):
        return []
    cks = []
    for p in Path(output_dir).glob("checkpoint-*"):
        try:
            step = int(p.name.split("-")[-1])
        except ValueError:
            step = -1
        cks.append((step, str(p)))
    cks.sort(key=lambda x: x[0])
    return cks

def latest_checkpoint(output_dir):
    cks = list_checkpoints(output_dir)
    return cks[-1][1] if cks else None

def print_resume_status(output_dir, label=""):
    """打印输出目录是否可续训（跑 A2/C2/C3 前可调用）。"""
    tag = label or output_dir
    cks = list_checkpoints(output_dir)
    state = os.path.join(output_dir, "trainer_state.json")
    print(f"=== resume 检查: {tag} ===")
    print("  output_dir:", output_dir, "->", os.path.isdir(output_dir))
    if cks:
        print(f"  checkpoints: {len(cks)} 个，最新 step={cks[-1][0]} -> {cks[-1][1]}")
        print("  将自动 resume（ENABLE_RESUME=True 时）")
    else:
        print("  checkpoints: 无 → 将从头训练")
    if os.path.isfile(state):
        try:
            with open(state) as f:
                st = json.load(f)
            print(f"  trainer_state: global_step={st.get('global_step')} max_steps={st.get('max_steps')}")
        except Exception as e:
            print("  trainer_state 读取失败:", e)
    print()

def build_train_cmd(output_dir, run_name, zero3_path, attn="flash_attention_2", resume_path=None):
    save_steps = int(globals().get("SAVE_STEPS", 10) or 0)
    logging_steps = int(globals().get("LOGGING_STEPS", 1) or 1)
    save_total_limit = int(globals().get("SAVE_TOTAL_LIMIT", 1) or 2)
    cmd = [
        "deepspeed", "--num_gpus=1", os.path.join(REPO, "tinyllava/train/train.py"),
        "--deepspeed", zero3_path,
        "--video_data_path", DATA_ROOT, "--video_folder", SMALL_JSONL,
        "--is_multimodal", "True", "--conv_version", "qwen2_base",
        "--model_name_or_path", "Qwen/Qwen2.5-3B",
        "--vision_tower", "google/siglip-so400m-patch14-384",
        "--connector_type", "groupresampler",
        "--num_frames", str(NUM_FRAMES), "--num_queries", str(NUM_QUERIES),
        "--mm_vision_select_layer", "-2", "--image_aspect_ratio", "square",
        "--attn_implementation", attn, "--bf16", "True",
        "--training_recipe", "common", "--tune_type_llm", "full",
        "--tune_type_vision_tower", "frozen", "--tune_vision_tower_from_layer", "0",
        "--tune_type_connector", "full", "--group_by_modality_length", "False",
        "--pretrained_model_path", CKPT,
        "--output_dir", output_dir,
        "--num_train_epochs", "1", "--max_steps", str(MAX_STEPS),
        "--per_device_train_batch_size", "1", "--gradient_accumulation_steps", "1",
        "--evaluation_strategy", "no",
        "--learning_rate", "5e-6", "--weight_decay", "0.0", "--warmup_ratio", "0.03",
        "--lr_scheduler_type", "cosine", "--logging_steps", str(logging_steps),
        "--tf32", "False", "--report_to", "none",
        "--model_max_length", str(MODEL_MAX_LENGTH), "--gradient_checkpointing", "True",
        "--dataloader_num_workers", "2", "--lazy_preprocess", "True", "--tokenizer_use_fast", "False",
        "--run_name", run_name,
        "--disable_tqdm", "False",
    ]
    if save_steps > 0:
        cmd += [
            "--save_strategy", "steps",
            "--save_steps", str(save_steps),
            "--save_total_limit", str(save_total_limit),
        ]
    else:
        cmd += ["--save_strategy", "no"]
    if resume_path:
        cmd += ["--resume_from_checkpoint", resume_path]
    return cmd

def _print_progress(step, total, t0, extra=""):
    step = max(0, min(int(step), int(total)))
    pct = 100.0 * step / max(total, 1)
    bar_len = 28
    filled = int(bar_len * step / max(total, 1))
    bar = "█" * filled + "░" * (bar_len - filled)
    elapsed = time.time() - t0
    eta = (elapsed / step * (total - step)) if step > 0 else float("nan")
    msg = f"[{bar}] {step}/{total} ({pct:5.1f}%)  elapsed={elapsed/60:.1f}m  ETA={eta/60:.1f}m"
    if extra:
        msg += f"  {extra}"
    print(f"\r{msg}", end="", flush=True)

def _parse_step_from_line(line, max_steps, last_step):
    """从 HF / tqdm / checkpoint 日志中解析当前 step。"""
    # tqdm: 12/50 or 12/50 [
    m = re.search(rf"\b(\d+)\s*/\s*{max_steps}\b", line)
    if m:
        return max(last_step, int(m.group(1)))
    # Saving model checkpoint to .../checkpoint-15
    m = re.search(r"checkpoint-(\d+)", line)
    if m and ("Saving" in line or "saving" in line or "checkpoint-" in line):
        return max(last_step, int(m.group(1)))
    # {'loss': ..., 'epoch': ...} 无法直接得 step；尝试 global_step
    m = re.search(r"['\"]global_step['\"]\s*:\s*(\d+)", line)
    if m:
        return max(last_step, int(m.group(1)))
    m = re.search(r"\bstep\s*[:=]\s*(\d+)\b", line, re.I)
    if m:
        val = int(m.group(1))
        if 0 < val <= max_steps:
            return max(last_step, val)
    return last_step

def run_training(output_dir, run_name):
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(OUT_BASE, exist_ok=True)
    zero3_path = ensure_zero3_config(REPO)

    resume_path = None
    if globals().get("ENABLE_RESUME", True):
        resume_path = latest_checkpoint(output_dir)
        if resume_path:
            print("发现 checkpoint，将 resume:", resume_path)
        else:
            print("无 checkpoint，从头训练:", output_dir)

    cmd = build_train_cmd(output_dir, run_name, zero3_path, resume_path=resume_path)
    env = os.environ.copy()
    env["PYTHONPATH"] = REPO + os.pathsep + env.get("PYTHONPATH", "")
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    env["PYTHONUNBUFFERED"] = "1"
    env["TQDM_MININTERVAL"] = "1"

    print("执行训练:", run_name, "->", output_dir)
    print("超参: g={}, max_steps={}, frames={}, save_steps={}, resume={}".format(
        NUM_GENERATIONS, MAX_STEPS, NUM_FRAMES,
        globals().get("SAVE_STEPS", 0), bool(resume_path)))
    print("(流式日志 + 进度条；Mac 请保持防休眠)\n")

    t0 = time.time()
    last_step = 0
    # 若 resume，用 checkpoint step 初始化进度
    if resume_path:
        try:
            last_step = int(Path(resume_path).name.split("-")[-1])
            _print_progress(last_step, MAX_STEPS, t0, extra="resumed")
            print()
        except Exception:
            pass

    p = subprocess.Popen(
        cmd, cwd=REPO, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end="")
        new_step = _parse_step_from_line(line, MAX_STEPS, last_step)
        if new_step > last_step:
            last_step = new_step
            _print_progress(last_step, MAX_STEPS, t0)
            print()
    rc = p.wait()
    if last_step:
        _print_progress(last_step, MAX_STEPS, t0, extra="done" if rc == 0 else "FAILED")
        print()
    if rc != 0:
        print("训练失败 exit=", rc, "| 若已有 checkpoint-*，修复后重跑本单元即可续训")
        raise SystemExit(rc)
    print("训练完成:", output_dir, f"| wall={(time.time()-t0)/60:.1f} min")

def apply_cppo_patch(pruning_rate: float):
    """覆盖 CPPO trainer，并设置 pruning_rate / generations。"""
    src_patch = os.path.join(MIXUP_REPO, "mixup", "patches", "tinyllava_trainer_reason_cppo.py")
    trainer_path = os.path.join(REPO, "tinyllava", "train", "tinyllava_trainer_reason.py")
    assert os.path.isfile(src_patch), f"缺少补丁: {src_patch}"
    shutil.copy(src_patch, trainer_path)
    with open(trainer_path, "r", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r"self\.num_generations = .*", f"self.num_generations = {NUM_GENERATIONS}  # Phase1", content, count=1)
    content = re.sub(r"self\.num_frame = .*", f"self.num_frame = {NUM_FRAME_TRAINER}", content, count=1)
    content = re.sub(
        r"self\.cppo_pruning_rate = .*",
        f"self.cppo_pruning_rate = {pruning_rate}  # CPPO Phase1",
        content,
        count=1,
    )
    with open(trainer_path, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"已应用 CPPO 补丁: pruning_rate={pruning_rate}, num_generations={NUM_GENERATIONS}")

print("工具函数就绪: ensure_zero3_config / build_train_cmd / run_training / apply_cppo_patch / print_resume_status")


---
## 7. 策略 A2：GRPO Baseline（g8，无剪枝）— P0

运行前确认 §5 已把 `cppo_pruning_rate=0`，且已重跑 §2 + §6。

输出：`outputs/grpo_A_g8`。

**当前续训**：若存在 `checkpoint-40`，将自动从 step 40 续跑至 `MAX_STEPS=50`。  
跑前可在 Drive 删除同目录下的 `checkpoint-30` / `checkpoint-35`，只留 40，并清空回收站。


In [ ]:
try:
    _ = REPO, OUT_A2, run_training, NUM_GENERATIONS, MAX_STEPS, print_resume_status
except NameError as e:
    raise RuntimeError("请先按顺序运行 §2 → §5 → §6") from e

print_resume_status(OUT_A2, "A2")
cks = list_checkpoints(OUT_A2)
if cks:
    print(f"将从最新 checkpoint-{cks[-1][0]} 续训 → 目标 MAX_STEPS={MAX_STEPS}")
else:
    print("未找到 checkpoint，将从头训练")
print(f"即将训练 A2 | g={NUM_GENERATIONS} | steps={MAX_STEPS} | save_steps={SAVE_STEPS} | out={OUT_A2}")
run_training(OUT_A2, "grpo_A_g8")


---
## 8. 策略 C2：CPPO p=0.5（g8）— P0

与 A2 **仅差** `cppo_pruning_rate=0.5`。输出：`outputs/grpo_CPPO_g8_p50`。

同样支持断点续训与进度条（依赖 §2 + §6）。


In [ ]:
print_resume_status(OUT_C2, "C2")
apply_cppo_patch(0.5)
print(f"即将训练 C2 | g={NUM_GENERATIONS} | steps={MAX_STEPS} | save_steps={SAVE_STEPS} | out={OUT_C2}")
run_training(OUT_C2, "grpo_CPPO_g8_p50")


---
## 9. 策略 C3：CPPO p=0.75（可选）

仅在 A2/C2 显示有加速且时间充裕时运行。断点续训 / 进度条同 A2/C2。


In [ ]:
# 可选：取消下行注释以运行 C3
# print_resume_status(OUT_C3, "C3")
# apply_cppo_patch(0.75)
# run_training(OUT_C3, "grpo_CPPO_g8_p75")
print("C3 默认跳过。取消注释以启用。")


---
## 10. 结果对比（填 schedule §8）

读取 `trainer_state.json`，打印相对 A2 的加速百分比，并输出可粘贴到 `schedule.md` 的 Markdown 行。


In [ ]:
import os
import json

def _read_last_metrics(out_dir):
    path = os.path.join(out_dir, "trainer_state.json")
    if not os.path.exists(path):
        return None
    with open(path, "r", encoding="utf-8") as f:
        state = json.load(f)
    log = state.get("log_history", [])
    last_with_loss, last_runtime = None, None
    for e in reversed(log):
        if last_with_loss is None and ("loss" in e or "reward" in e):
            last_with_loss = e
        if last_runtime is None and "train_runtime" in e:
            last_runtime = e
        if last_with_loss and last_runtime:
            break
    return {"metrics": last_with_loss, "runtime": last_runtime}

try:
    _ = OUT_A2, OUT_C2, OUT_C3
except NameError:
    OUT_BASE = "/content/drive/MyDrive/MixUpLLaVA-video-r1/outputs"
    OUT_A2 = os.path.join(OUT_BASE, "grpo_A_g8")
    OUT_C2 = os.path.join(OUT_BASE, "grpo_CPPO_g8_p50")
    OUT_C3 = os.path.join(OUT_BASE, "grpo_CPPO_g8_p75")

rows = []
baseline_rt = None
for name, d, tag in [
    ("A2 GRPO baseline (g8)", OUT_A2, "A2"),
    ("C2 CPPO p=0.5 (g8)", OUT_C2, "C2"),
    ("C3 CPPO p=0.75 (g8)", OUT_C3, "C3"),
]:
    data = _read_last_metrics(d)
    print(f"--- {name} ---")
    print("  目录:", d, "| 存在:", os.path.isdir(d))
    if not data:
        print("  无 trainer_state.json")
        rows.append({"tag": tag, "name": name, "dir": d})
        continue
    m, r = data["metrics"], data["runtime"]
    rt = r.get("train_runtime") if r else None
    if tag == "A2" and rt:
        baseline_rt = rt
    row = {
        "tag": tag,
        "name": name,
        "step": m.get("step") if m else None,
        "loss": m.get("loss") if m else None,
        "reward": m.get("reward") if m else None,
        "reward_std": m.get("reward_std") if m else None,
        "train_runtime": rt,
        "steps_per_second": r.get("train_steps_per_second") if r else None,
    }
    if baseline_rt and rt and tag != "A2":
        row["speedup_vs_A2_%"] = round((baseline_rt - rt) / baseline_rt * 100, 2)
    rows.append(row)
    if m:
        print(f"  step={m.get('step')} loss={m.get('loss')} reward={m.get('reward')}")
    if r:
        print(f"  runtime={rt}s steps/s={r.get('train_steps_per_second')}")
        if "speedup_vs_A2_%" in row:
            print(f"  相对 A2 加速: {row['speedup_vs_A2_%']}%")

print("\n=== 可粘贴到 schedule.md §8 ===")
print("| 策略 | 输出目录 | max_steps | train_runtime(s) | steps/s | reward | 相对 A2 加速 |")
print("|------|----------|-----------|------------------|---------|--------|--------------|")
for row in rows:
    tag = row.get("tag", "")
    out = {"A2": "grpo_A_g8", "C2": "grpo_CPPO_g8_p50", "C3": "grpo_CPPO_g8_p75"}.get(tag, "")
    rt = row.get("train_runtime", "")
    sps = row.get("steps_per_second", "")
    rew = row.get("reward", "")
    sp = row.get("speedup_vs_A2_%", "—" if tag == "A2" else "")
    ms = row.get("step", "")
    print(f"| {tag} | `{out}` | {ms} | {rt} | {sps} | {rew} | {sp} |")


---
## 附录

- **公平对比**：A2 与 C2/C3 仅差 `cppo_pruning_rate`；勿叠加策略 B reward。
- **断点续训**：`SAVE_STEPS=10` 写 `checkpoint-*`；`SAVE_TOTAL_LIMIT=1` 只留最新。断连后重跑 §0–§6 再跑对应训练单元。
- **云盘**：删除的 checkpoint 会进回收站并继续占配额 → **务必清空回收站**。
- **进度**：日志流式打印；解析到 `n/MAX_STEPS` 或 `checkpoint-n` 时更新进度条。
- **OOM**：`NUM_GENERATIONS=4` 或减小 `MAX_STEPS`。
- **息屏**：Mac 用 `caffeinate -dims`；保持 Colab 标签页。
